# Day 4 動手練習：有沒有 Chain-of-Thought，答案真的不一樣嗎？

搭配 [day04.md](./day04.md)。文章裡提到 Zero-shot Chain-of-Thought：不用給範例，只要在題目後面加一句「請一步一步思考」，就可能讓模型在多步驟問題上表現變好。

這份 notebook 用一個很小的地端模型（**Qwen2.5-0.5B-Instruct**，約 5 億參數，跟 Day 2 用的是同一個模型），對同一批題目分別用「直接回答」和「Chain-of-Thought（CoT）」兩種問法各問一次，把完整的推理過程印出來，再對照正確答案，看兩種問法的正確率差多少。

**先講結論，等一下你會在自己的機器上重新跑出一次**：這個小模型直接回答 3 題全錯——這件事無論在哪台機器上重跑，測試下來都很穩定。只在題目後面加一句「請一步一步思考」，CoT 版本原本可以 3 題全對。

但這份 notebook 的每個 prompt 還多加了一句「請用繁體中文回答」（原因見下方說明），這句話一入場，CoT 就不再穩定全對——通常還是會贏過直接回答，但確切是哪一題出錯、錯成什麼答案，會因為執行的機器、甚至只是提示語調整一兩個字而不同。我自己在準備這份 notebook 的過程中，同一種題型就看過好幾種不同的錯誤版本（有時是加減方向搞反，有時是把「大 3 歲」誤解成「3 倍」）。這不是我沒調好，而是這份 notebook 想讓你看到的重點：一個只有幾億參數的小模型，推理過程有多容易被「跟任務本身無關」的細節牽動。後面會再用一題「不需要推理」的簡單問題，進一步說明 CoT 不是萬用魔法。**你自己跑出來的具體數字不需要跟這裡寫的一樣，那才是正常的——重點是看下面印出來的實際過程，不是對答案。**

## 環境安裝

In [1]:
%pip install -q torch transformers accelerate pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import time

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"使用裝置: {device}")

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
print("正在載入小模型...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()
print("載入完成！")

使用裝置: mps
正在載入小模型...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

載入完成！


---
## Part 1｜三題「需要拆步驟」的應用題

三題都需要至少兩個步驟才能算出答案，跟 day04.md 裡「打折加稅金」的例子是同一種類型，換了新的數字與情境。每題都先手動算出正確答案，等一下用來對照模型的回答是否正確。

In [3]:
questions = [
    {
        "id": "A",
        "text": "一件商品原價 1,200 元，先打七折，再加上折後價格的 5% 服務費，最後要付多少元？",
        "answer": "882",
    },
    {
        "id": "B",
        "text": "小華比小美大 3 歲，兩人年齡加起來是 25 歲，請問小華幾歲？",
        "answer": "14",
    },
    {
        "id": "C",
        "text": "一台印表機每分鐘印 15 張，另一台每分鐘印 10 張，兩台一起印，印完 200 張需要幾分鐘？",
        "answer": "8",
    },
]

for q in questions:
    print(f"[{q['id']}] {q['text']}  （正確答案：{q['answer']}）")

[A] 一件商品原價 1,200 元，先打七折，再加上折後價格的 5% 服務費，最後要付多少元？  （正確答案：882）
[B] 小華比小美大 3 歲，兩人年齡加起來是 25 歲，請問小華幾歲？  （正確答案：14）
[C] 一台印表機每分鐘印 15 張，另一台每分鐘印 10 張，兩台一起印，印完 200 張需要幾分鐘？  （正確答案：8）


接著寫兩個生成函式：`ask_direct` 要求模型只給答案，不要說明過程；`ask_cot` 則在題目後面加一句「請一步一步思考」，並要求最後一行用「答案：」標示結論。兩者都用貪婪解碼（`do_sample=False`），理論上同一個提示應該每次都得到一樣的結果。實際測試下來，A/B/C 這三題每次重跑都很穩定；但比較長、比較開放的生成（像等一下 Part 2 的例子），在 Apple Silicon 的 MPS 加速上偶爾會因為浮點運算順序的微小差異，讓重跑結果不完全相同。這也是個提醒：就算用了「貪婪解碼」，不同硬體、不同時間點重新產生的文字，也不保證每個字都一樣。

兩個 prompt 最後都多加了一句「請用繁體中文回答」。原因是這個模型的訓練資料裡簡體中文比較多，不加這句話，輸出常常會不自覺夾雜「我们」「这个」之類的簡體字。加了之後確實乾淨很多，但**代價是 CoT 的正確率會下降**——下面會用實際數字呈現這個代價有多大。這件事本身也是個小教材：對一個只有幾億參數的小模型來說，prompt 裡任何一句話，就算立意單純（只是要求輸出的文字型態），都可能牽動它原本就不太穩固的推理路徑。

In [4]:
def generate(prompt, max_new_tokens):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.time() - start

    new_tokens = output.shape[1] - inputs["input_ids"].shape[1]
    answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return answer.strip(), elapsed, new_tokens


def ask_direct(question):
    prompt = f"{question}\n請直接只回答最終答案，不要說明過程。請用繁體中文回答。"
    return generate(prompt, max_new_tokens=30)


def ask_cot(question):
    prompt = f"{question}\n請一步一步思考，寫出你的推理過程，最後一行用「答案：」加上最終答案。請用繁體中文回答。"
    return generate(prompt, max_new_tokens=250)


def extract_number(text):
    # 優先抓「答案：123」這種明確標示；抓不到就退而求其次，抓文字裡最後出現的數字
    m = re.search(r"答案\s*[:：是]?\s*(-?\d+)", text)
    if m:
        return m.group(1)
    nums = re.findall(r"-?\d+", text)
    return nums[-1] if nums else None

逐題執行，把兩種問法的完整輸出都印出來——這就是題目要求的「把過程 print 出來」。CoT 那欄可以看到模型實際上怎麼拆解問題。

In [5]:
results = []

for q in questions:
    print("=" * 70)
    print(f"題目 [{q['id']}]：{q['text']}")
    print(f"正確答案：{q['answer']}")

    direct_text, direct_time, direct_tokens = ask_direct(q["text"])
    direct_pred = extract_number(direct_text)
    print(f"\n【直接回答】（{direct_time:.2f}s, {direct_tokens} tokens）")
    print(direct_text)
    print(f"→ 模型答案：{direct_pred}　{'✅ 正確' if direct_pred == q['answer'] else '❌ 錯誤'}")

    cot_text, cot_time, cot_tokens = ask_cot(q["text"])
    cot_pred = extract_number(cot_text)
    print(f"\n【Chain-of-Thought】（{cot_time:.2f}s, {cot_tokens} tokens）")
    print(cot_text)
    print(f"→ 模型答案：{cot_pred}　{'✅ 正確' if cot_pred == q['answer'] else '❌ 錯誤'}")

    results.append({
        "題目": q["id"],
        "正確答案": q["answer"],
        "直接回答": direct_pred,
        "直接正確？": direct_pred == q["answer"],
        "CoT 回答": cot_pred,
        "CoT 正確？": cot_pred == q["answer"],
        "直接耗時(s)": round(direct_time, 2),
        "CoT 耗時(s)": round(cot_time, 2),
    })
    print()

題目 [A]：一件商品原價 1,200 元，先打七折，再加上折後價格的 5% 服務費，最後要付多少元？
正確答案：882

【直接回答】（0.32s, 5 tokens）
700元
→ 模型答案：700　❌ 錯誤

【Chain-of-Thought】（3.40s, 127 tokens）
首先，商品原價為1,200元。

接著，先打七折，即打70%，因此折扣後的价格是：
\[ 1,200 \times 0.7 = 840\,元 \]

然後，加上折後價格的5%服務費，即：
\[ 840 \times (1 + 0.05) = 840 \times 1.05 = 882\,元 \]

所以，最後要付的總价钱是882元。

答案：882元
→ 模型答案：882　✅ 正確

題目 [B]：小華比小美大 3 歲，兩人年齡加起來是 25 歲，請問小華幾歲？
正確答案：14

【直接回答】（0.13s, 4 tokens）
18岁
→ 模型答案：18　❌ 錯誤

【Chain-of-Thought】（5.44s, 241 tokens）
要解這個問題，我們可以按照以下步驟來思維：

1. **已知條件**：
   - 小華比小美大3歲。
   - 二人年齡的總和是25歲。

2. **設立方程**：
   假設小華的年龄為 \( x \) 岁，小美的 age 就是 \( x + 3 \) 岁（因為小華比小美大3歲）。

3. **代入已知數值**：
   根據題目，兩人的年齡之和是25歲，所以有等式：
   \[
   x + (x + 3) = 25
   \]

4. **解方程**：
   \[
   2x + 3 = 25
   \]
   \[
   2x = 22
   \]
   \[
   x = 11
   \]

因此，小華的年齡是11歲。

根據以上推理過程，答案是：小華是11歲。
→ 模型答案：11　❌ 錯誤

題目 [C]：一台印表機每分鐘印 15 張，另一台每分鐘印 10 張，兩台一起印，印完 200 張需要幾分鐘？
正確答案：8

【直接回答】（0.10s, 4 tokens）
60秒
→ 模型答案：60　❌ 錯誤

【Chain-of-Thought】（3.48s, 217 tokens）
要解決這個問題，我們可以先計算兩台印表機一起工作時的效

---
## 整理成表格對照

In [6]:
import pandas as pd

df = pd.DataFrame(results)
display(df)

direct_correct = df["直接正確？"].sum()
cot_correct = df["CoT 正確？"].sum()
print(f"\n直接回答正確率：{direct_correct}/{len(df)}")
print(f"CoT 回答正確率：{cot_correct}/{len(df)}")

,題目,正確答案,直接回答,直接正確？,CoT 回答,CoT 正確？,直接耗時(s),CoT 耗時(s)
0,A,882,700,False,882,True,0.32,3.40
1,B,14,18,False,11,False,0.13,5.44
2,C,8,60,False,8,True,0.10,3.48



直接回答正確率：0/3
CoT 回答正確率：2/3


**看數字之前先提醒自己**：這只有 3 題，是一次示範，不是嚴謹的統計實驗。而且**這裡印出來的正確率，你重跑一次不一定會拿到完全一樣的數字**——這不是 bug，是這份 notebook 想讓你實際遇到的現象。

**CoT 的正確率不再穩定停在 3/3，是因為 prompt 裡多了「請用繁體中文回答」這句話。** 拿掉這句話重新測試，CoT 可以穩定 3 題全對；加回去之後，CoT 通常還是會答對其中一兩題，但確切是哪一題出錯、錯誤的推理長什麼樣子，會隨著執行的機器、甚至提示語只調整一兩個字而改變——這是 Apple 的 MPS 加速在浮點運算上本來就不保證每台硬體都得到位元級相同結果所致，不是誰跑錯了。準備這份 notebook 時，我自己就遇過好幾種不同的組合：有時是加減方向搞反，有時是把倍數關係跟差距關係搞混，出錯的題目也不一定是同一題。

真正穩定、可以放心比較的是：**直接回答不管加不加這句話、不管在哪台機器上都是 0/3**。CoT 依然比直接回答可靠，只是這份「可靠」本身，比我們直覺以為的更容易被 prompt 裡不相干的細節動搖——包括你自己重跑這一格，很可能會拿到跟這裡不完全一樣的錯誤內容，這也是同一件事的展現，不用特地跟這份文字對答案。

---
## Part 2｜換一題不需要推理的簡單問題

day04.md 提醒過：「對簡單的知識查詢……硬要求模型長篇推理，可能只會讓回答更慢、更長，甚至在原本簡單的問題上多繞幾圈。」

這裡問一題單純的知識性問題，直接回答應該就很夠了，看看 CoT 版本會發生什麼事。

In [7]:
simple_q = "地球上面積最大的海洋是哪一個？"

direct_text, direct_time, direct_tokens = ask_direct(simple_q)
cot_text, cot_time, cot_tokens = ask_cot(simple_q)

print(f"題目：{simple_q}\n")
print(f"【直接回答】（{direct_time:.2f}s, {direct_tokens} tokens）")
print(direct_text)
print(f"\n【Chain-of-Thought】（{cot_time:.2f}s, {cot_tokens} tokens）")
print(cot_text)

題目：地球上面積最大的海洋是哪一個？

【直接回答】（0.12s, 2 tokens）
太平洋

【Chain-of-Thought】（3.68s, 232 tokens）
要找出地球上的積最大海洋，我們需要先了解地球的地理結構和海洋分布。

1. 地球表面主要由陆地和海洋组成。
2. 海洋主要分布在赤道附近，因为这是太阳辐射最强的地方。
3. 除了赤道附近的海洋外，还有许多位于两极的冰盖和高山地区，这些区域由于长期的低温环境，海水密度较大，因此积聚了大量水体。

基于以上信息，我们可以推測地球上的積最大海洋可能在南极洲或北极地区。然而，由于南极洲和北极地区的地理位置相对偏远且面积较小，它们的海洋面积通常不会超过其他大洲的大海面积。

综合考虑，最有可能的是在南极洲，因为它不仅有广阔的内陆平原（如西风带），而且其周围海域广阔，适合形成巨大的海洋系统。此外，南极洲的冰盖覆盖面积也很大，可以显著增加海洋体积。

所以，根據上述分析，答案應該是：

答案：南极洲

因為它具有广阔的内陆平原、丰富的冰盖和广阔的海洋空间，能夠形成巨大的海洋系统。


**觀察重點**：直接回答通常又快又短；CoT 版本不但可能花上好幾倍的字數與時間，答案本身也不保證正確——這題實測下來，CoT 大多能正確推出「太平洋」，但也可能一路推理到「南極洲」這種明顯錯誤的結論。推理過程讀起來一樣頭頭是道（先講地理結構、再講海水密度、最後「綜合考慮」），卻在中途悄悄把方向帶偏，最後仍然很有自信地端出一個錯的答案。這正是 day04.md 說的「看似合理、實際上無關的步驟」——差別在於，這次連答案都被這些「看似合理」的步驟拖著走偏了，不再只是「答案對、過程有點瑕疵」那麼溫和。

（如果你重跑這一格，直接回答這次甚至可能連格式都跑掉、答非所問；CoT 也可能出現像「南極洲」這種完全跑題的答案——這些都是真實會發生的情況，不是我刻意挑出來的反例。多跑幾次，感受一下同一個小模型在「不需要推理的問題」上有多不穩定：不只是推理內容不穩，連最後的結論都可能不穩。）

你可能也會注意到，即使 prompt 已經要求「請用繁體中文回答」，CoT 的內容裡偶爾還是會冒出一兩個簡體字。這句指示能大幅減少簡體字，但沒辦法保證百分之百——對這麼小的模型來說，「輸出語言」本身也只是一個機率上被拉高的傾向，不是一條寫死的規則。

---
## 停下來想一想

呼應 day04.md 的觀察問題，你可以照著這幾點重新看一次上面印出來的推理過程：

1. 模型把應用題拆成了哪些步驟？跟你自己心算的順序一樣嗎？
2. CoT 的推理過程中，有沒有出現多餘、甚至互相矛盾的句子？
3. 在需要多步驟計算的題目上，CoT 明顯比直接回答可靠；但在「地球上最大的海洋」這種單純知識題上，CoT 值得嗎？
4. 如果把三題應用題的數字改得更複雜（例如三層折扣、多個未知數），你覺得直接回答與 CoT 的正確率差距會變大還是變小？
5. 這份實驗只用了一個 5 億參數的小模型、3+1 道題目。如果換成更大的模型，兩種問法的差距還會這麼明顯嗎？
6. 「請用繁體中文回答」讓 CoT 正確率從 3/3 掉到 1/3。如果你的產品 prompt 裡也疊了一堆「輸出格式要求」「語氣要求」「長度限制」，這些看似跟任務本身無關的指示，會不會也在悄悄影響模型的正確率？

**Chain-of-Thought 讓模型多寫了一份計算過程，也讓我們多了一份可以檢查的東西——但看得到，不等於看到的就一定是對的。而 prompt 裡的每一句話，就算立意良善，對小模型來說都不是免費的。**